# `parse_question` — Brique 2 : parser une question utilisateur

Référence : chapitre **`docs/06_question_parsing.md`** (Kezhan).

## Sortie

**Toujours une `list[dict]`** (drop-in compat avec `src.question.understand_question`). Chaque entrée :

| Section | Contenu |
|---|---|
| `retrieval`  | `main_query`, `page_hint`, `section_hint`, `layout_hint`, `document_hint`, `anchor_keywords` |
| `generation` | `original_question`, `format_constraint`, `disambiguation`, `must_distinguish` |
| `_meta`      | `intent`, `document_type`, `bricks_active` (traçabilité) |

**Le JSON ne contient QUE des champs populés** (pas de `null`).

## Connaissance métier dans les PRESETS

| `document_type` | Briques actives |
|---|---|
| `pdf`   | toutes |
| `word`  | sans `page_hint` (pas de pagination stable en .docx) |
| `excel` | sans `page_hint` ni `section_hint` (axes orthogonaux : feuilles + colonnes) |
| `email` | sans `page_hint` ni `section_hint` |
| `pptx`  | toutes (slide = page) |

Override fin par appel : `parse_question(q, enable={"page_hint": False})`.

## Approche

Regex / heuristique uniquement (pas de LLM, conforme à `CLAUDE.md`).

In [ ]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ../../../..

## 1 — Démonstration sur une banque de questions

In [ ]:
import json
from docpipeline.question_parsing import parse_question

QUESTIONS = [
    "Quelle est la date d'effet du contrat ? Page 1, format YYYY-MM-DD.",
    "What is the limit per claim, not the deductible, in the Limits section page 5? Article L131-1 applies.",
    "Look in the exclusions section for the flooding clause.",
    "Compare the indemnification and liability caps in the latest version.",
    "List all the obligations of the seller, in bullet list.",
    "Find the policy number — it's usually in the header, format JSON.",
    "Dans le tableau page 3 du contrat MAIF, quel est le montant ? En euros.",
    "Combien coûte l'assurance ?",
]

for q in QUESTIONS:
    plan = parse_question(q, document_type='pdf')
    print(f'Q: {q}')
    print(json.dumps(plan, indent=2, ensure_ascii=False))
    print('-' * 80)

## 2 — Banque de cas (test visuel) avec sortie attendue

Couvre les bugs de `src/question/` initial qu'on a corrigés ici :
1. **`section_hint`** : `« Look in the exclusions section ... »` doit donner `"exclusions"` (et non `"for the flooding clause"`).
2. **`anchor_keywords`** : `« Article L131-1 »` doit apparaître dans les keywords.
3. **Pas de faux positif** sur les format-placeholders : `« YYYY-MM-DD »` ne doit PAS finir dans `anchor_keywords`.
4. **Intent `extract`** : `« What is ... »` doit être classé `extract` (et non `yes_no`).
5. **Document-aware** : sur un Word, `« page 3 »` doit être ignoré (preset Word n'a pas `page_hint`).

In [ ]:
from docpipeline.question_parsing import parse_question

CHECKS = [
    {
        'q': 'Look in the exclusions section for the flooding clause.',
        'doc': 'pdf',
        'check': lambda p: p[0]['retrieval'].get('section_hint') == 'exclusions',
        'desc': 'section_hint = "exclusions" (pas "for the flooding clause")',
    },
    {
        'q': 'Does article L131-1 of the insurance code apply here?',
        'doc': 'pdf',
        'check': lambda p: any('L131-1' in k or '131-1' in k for k in p[0]['retrieval'].get('anchor_keywords', [])),
        'desc': 'anchor_keywords contient L131-1',
    },
    {
        'q': 'Date au format YYYY-MM-DD',
        'doc': 'pdf',
        'check': lambda p: 'YYYY' not in p[0]['retrieval'].get('anchor_keywords', []),
        'desc': 'YYYY exclu des anchor_keywords (faux positif évité)',
    },
    {
        'q': 'What is the maximum coverage amount?',
        'doc': 'pdf',
        'check': lambda p: p[0]['_meta']['intent'] == 'extract',
        'desc': 'intent = extract (pas yes_no)',
    },
    {
        'q': 'Plafond page 3 du contrat',
        'doc': 'word',
        'check': lambda p: 'page_hint' not in p[0]['retrieval'],
        'desc': 'En Word, page_hint ignoré (preset Word sans page_hint)',
    },
    {
        'q': 'Le plafond, pas la franchise',
        'doc': 'pdf',
        'check': lambda p: any('franchise' in d for d in p[0]['generation'].get('must_distinguish', [])),
        'desc': 'disambiguation détecte « franchise » comme distractor',
    },
    {
        'q': 'Combien coute l\'assurance ?',
        'doc': 'pdf',
        'check': lambda p: 'null' not in __import__('json').dumps(p),
        'desc': 'JSON sans null sur question minimale',
    },
]

for c in CHECKS:
    plan = parse_question(c['q'], document_type=c['doc'])
    ok = c['check'](plan)
    flag = 'OK  ' if ok else 'FAIL'
    print(f"[{flag}] {c['desc']}")
    print(f"       Q: {c['q']}  (doc_type={c['doc']})")